In [0]:
CREATE OR REPLACE VIEW airline_catalog.semantic.vw_monthly_performance AS -- Crear o reemplazar la vista de desempeño mensual

WITH monthly_metrics AS -- CTE para calcular métricas mensuales
(
    SELECT

        d.year, -- Año del vuelo
        d.month, -- Mes del vuelo

        COUNT(*) AS total_flights, -- Total de vuelos en el mes

        SUM(CASE
                WHEN ff.flight_status = 'On Time' THEN 1
                ELSE 0
            END) AS on_time_flights, -- Total de vuelos a tiempo

        SUM(CASE
                WHEN ff.flight_status = 'Delayed' THEN 1
                ELSE 0
            END) AS delayed_flights, -- Total de vuelos retrasados

        SUM(CASE
                WHEN ff.cancelled THEN 1
                ELSE 0
            END) AS cancelled_flights, -- Total de vuelos cancelados

        ROUND(AVG(ff.departure_delay),2) AS avg_departure_delay, -- Promedio de retraso en salida

        ROUND(AVG(ff.arrival_delay),2) AS avg_arrival_delay, -- Promedio de retraso en llegada

        ROUND(AVG(ff.distance),2) AS avg_distance, -- Promedio de distancia recorrida

        ROUND(SUM(ff.distance),2) AS total_distance, -- Distancia total recorrida

        ROUND(
            100.0 *
            SUM(CASE WHEN ff.flight_status='Delayed' THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS pct_delayed, -- Porcentaje de vuelos retrasados

        ROUND(
            100.0 *
            SUM(CASE WHEN ff.cancelled THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS pct_cancelled -- Porcentaje de vuelos cancelados

    FROM airline_catalog.gold.fact_flights ff -- Tabla de hechos de vuelos

    INNER JOIN airline_catalog.gold.dim_date d -- Unión con la dimensión de fechas
        ON ff.date_id = d.date_id -- Relación por fecha

    GROUP BY

        d.year, -- Agrupar por año
        d.month -- Agrupar por mes
)

SELECT

    *, -- Seleccionar todas las métricas calculadas

    RANK() OVER (
        ORDER BY total_flights DESC
    ) AS traffic_rank, -- Ranking de tráfico mensual

    RANK() OVER (
        ORDER BY avg_arrival_delay ASC
    ) AS punctuality_rank, -- Ranking de puntualidad mensual

    LAG(total_flights) OVER (
        ORDER BY year, month
    ) AS previous_month_flights, -- Total de vuelos del mes anterior

    total_flights -
    LAG(total_flights) OVER (
        ORDER BY year, month
    ) AS flight_variation -- Variación de vuelos respecto al mes anterior

FROM monthly_metrics; -- Fuente de métricas mensuales

In [0]:
select * from airline_catalog.semantic.vw_monthly_performance